<a href="https://colab.research.google.com/github/p3nny-lan3/DATA205-Final-Project-Fall-2026/blob/main/ingestion/notebooks/01_Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyzing Occupational Wage Gaps Across Maryland
### Rin Hwang
### DATA 205 Capstone in Data Science

## Part 1: Ingestion and Cleaning

### Cleaning and Wrangling the First Dataset, Maryland Occupational Statistics in 2025

In [1]:
import pandas as pd
import numpy as np

I wanted to use a raw link to download my data and list all the sheet names, however on Github, my file was too big. Thus, I used Gemini AI to help retrieve a raw link. Below is the code for that.

In [2]:
import io
import requests
!pip install openpyxl

wagesdata = "https://media.githubusercontent.com/media/p3nny-lan3/DATA205-Final-Project-Fall-2026/main/data/raw/2025_Maryland_OEWS.xlsx"

# Download the binary data from GitHub
response = requests.get(wagesdata)
response.raise_for_status()

# Wrap the binary content into an in-memory byte stream
excel_data = io.BytesIO(response.content)

# Viewing sheet names
xls = pd.ExcelFile(excel_data, engine="openpyxl")
sheet_names = xls.sheet_names

print("Sheet names in dataset:")
for name in sheet_names:
    print(f"- {name}")

Sheet names in dataset:
- About the Numbers
- All Areas
- Statewide
- Anne Arundel
- Baltimore City
- Baltimore County
- Carroll
- Frederick
- Howard
- Lower Shore
- Montgomery
- Prince Geroge's
- Southern Maryland
- Susquehanna
- Upper Shore
- Western Maryland


In [3]:
# I will only be using 'All Areas'sheet that includes all Local Workforce Development Areas (LWDA) as well as statewide data

df_wages_all = pd.read_excel(xls, sheet_name="All Areas")

df_wages_all.head()
df_wages_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6888 entries, 0 to 6887
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Period Year             6888 non-null   int64  
 1   Area Name               6888 non-null   object 
 2   Occupation Code         6888 non-null   object 
 3   Occupation Name         6888 non-null   object 
 4   Occupation Group        6888 non-null   object 
 5   Employment              6792 non-null   float64
 6   Hourly Mean Wage        6564 non-null   float64
 7   Annual  Mean Wage       6864 non-null   float64
 8   Hourly Entry Wage       6564 non-null   float64
 9   Annual Entry Wage       6864 non-null   float64
 10  Hourly Experience Wage  6534 non-null   float64
 11  Annual Experience Wage  6834 non-null   float64
 12  Hourly 10th Percentile  6564 non-null   float64
 13  Annual 10th Percentile  6864 non-null   float64
 14  Hourly 25th Percentile  6564 non-null   

In [4]:
areas_list_demo = df_wages_all["Area Name"].unique()

print(areas_list_demo)

['Maryland' 'Anne Arundel LWDA' 'Baltimore City LWDA'
 'Baltimore County LWDA' 'Carroll LWDA' 'Frederick LWDA' 'Howard LWDA'
 'Lower Shore LWDA' 'Montgomery LWDA' "Prince George's LWDA"
 'Southern Maryland LWDA' 'Susquehanna LWDA' 'Upper Shore LWDA'
 'Western Maryland LWDA']


In [20]:
# Filter out hourly columns
df_clean = df_wages_all.loc[:, ~df_wages_all.columns.str.startswith("Hourly")].copy()
df_clean = df_clean.dropna(subset=["Annual  Mean Wage", "Annual Median Wage"], how="all")

# Format and standardize column names
df_clean.columns = (
    df_clean.columns.str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.strip("_")
)

# Drop year column since all is already 2025
df_clean = df_clean.drop(columns=["period_year"])

# Cast data types
annual_cols = df_clean.filter(like="annual").columns
df_clean[annual_cols] = df_clean[annual_cols].apply(pd.to_numeric, errors="coerce")
df_clean["employment"] = pd.to_numeric(df_clean["employment"], errors="coerce")

# Create hierarchy flags for occupation codes
df_clean["is_total"] = df_clean["occupation_code"] == "00-0000"
df_clean["is_major_group"] = (
    df_clean["occupation_code"].str.endswith("0000") & ~df_clean["is_total"]
)
df_clean["is_detailed"] = (
    ~df_clean["occupation_code"].str.endswith("0000") & ~df_clean["is_total"]
)

# Export to CSV
df_clean.to_csv("MD_OEWS_Cleaned.csv", index=False)

In [21]:
# Download clean dataset
# Comment out if already ran and downloaded

from google.colab import files
files.download("MD_OEWS_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Check for missing values
df_clean.isna().sum()

,0
area_name,0
occupation_code,0
occupation_name,0
occupation_group,0
employment,96
annual_mean_wage,0
annual_entry_wage,0
annual_experience_wage,30
annual_10th_percentile,0
annual_25th_percentile,0


The missing values are just standard state data suppressions for tiny sample sizes in employment and experience, which doesn't affect the analysis since all core annual median wages have no missing values.

### Cleaning and Wrangling the Second Dataset, Maryland Demographics

I also used Gemini AI for this dataset to help retrieve a raw link, since the file was too big. Below is the code for that.

In [23]:
import io
import os
import pandas as pd
import requests

url = "https://media.githubusercontent.com/media/p3nny-lan3/DATA205-Final-Project-Fall-2026/main/data/raw/Maryland_Regional_Demographics.xlsx"
response = requests.get(url)
response.raise_for_status()

# Load the 'Regional_Share' sheet (header is on row 2, skipping agency title rows)
df_raw = pd.read_excel(
    io.BytesIO(response.content),
    sheet_name="Regional_Share",
    header=2,
    engine="openpyxl",
)

# Selecting only the four columns I need
cols = ["Jurisdiction", "Sex", "Race", "2025"]
df = df_raw[cols].copy()

# Drop any missing values
df = df.dropna().reset_index(drop=True)

# Filter out female and male rows and just keep total population
df = df[df["Sex"].str.strip() == "Total"].copy()

# Standardize column names (lowercase with underscores)
df.columns = (
    df.columns.astype(str)
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.strip("_")
)

# Trim trailing whitespace from string columns
df["jurisdiction"] = df["jurisdiction"].astype(str).str.strip()
df["sex"] = df["sex"].astype(str).str.strip()
df["race"] = df["race"].astype(str).str.strip()

# Cast population into integer
if df["2025"].dtype == object:
    df["2025"] = df["2025"].str.replace(",", "", regex=False)
df["2025"] = pd.to_numeric(df["2025"], errors="coerce").astype(int)

# Export to csv
df.to_csv("MD_Demographics_Cleaned.csv", index=False)

In [24]:
# Download clean dataset
# Comment out if already ran and downloaded

from google.colab import files
files.download("MD_Demographics_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import pandas as pd

# Load cleaned demographics dataset
df_demo = pd.read_csv("MD_Demographics_Cleaned.csv")

# Geographic mapping: 24 jurisdictions -> 13 local workforce development areas (LWDAs) and Statewide
lwda_mapping = {

    # Single-County LWDAs
    "Anne Arundel County": "Anne Arundel LWDA",
    "Baltimore City": "Baltimore City LWDA",
    "Baltimore County": "Baltimore County LWDA",
    "Carroll County": "Carroll LWDA",
    "Frederick County": "Frederick LWDA",
    "Howard County": "Howard LWDA",
    "Montgomery County": "Montgomery LWDA",
    "Prince George's County": "Prince George's LWDA",

    # Multi-County Regional LWDAs
    "Allegany County": "Western Maryland LWDA",
    "Garrett County": "Western Maryland LWDA",
    "Washington County": "Western Maryland LWDA",
    "Caroline County": "Upper Shore LWDA",
    "Dorchester County": "Upper Shore LWDA",
    "Kent County": "Upper Shore LWDA",
    "Queen Anne's County": "Upper Shore LWDA",
    "Talbot County": "Upper Shore LWDA",
    "Somerset County": "Lower Shore LWDA",
    "Wicomico County": "Lower Shore LWDA",
    "Worcester County": "Lower Shore LWDA",
    "Calvert County": "Southern Maryland LWDA",
    "Charles County": "Southern Maryland LWDA",
    "St. Mary's County": "Southern Maryland LWDA",
    "Cecil County": "Susquehanna LWDA",
    "Harford County": "Susquehanna LWDA",

    # Statewide
    "Maryland": "Maryland",
}

# Engineer area names
df_demo["area_name"] = df_demo["jurisdiction"].map(lwda_mapping)

# Filter for Total Population and Minority Population categories
target_races = [
    "Total Population",
    "Total Racial and Ethnic Minority Population **",
]
df_target_pop = df_demo[df_demo["race"].isin(target_races)].copy()

# Pivot so 'Total Population' and 'Minority' become columns
df_pivoted = df_target_pop.pivot_table(
    index="area_name", columns="race", values="2025", aggfunc="sum"
).reset_index()

# Clean column names
df_lwda = df_pivoted.rename(
    columns={
        "Total Population": "total_population",
        "Total Racial and Ethnic Minority Population **": "minority_population",
    }
)
df_lwda.columns.name = None

# Calculate minority share percentage
df_lwda["minority_pct"] = (
    df_lwda["minority_population"] / df_lwda["total_population"] * 100
).round(2)

# Export to csv
df_lwda.to_csv("MD_LWDA_Demographics_Summary.csv", index=False)
print(df_lwda)

                 area_name  total_population  minority_population  \
0        Anne Arundel LWDA            603380               246056   
1      Baltimore City LWDA            569997               420068   
2    Baltimore County LWDA            847650               432903   
3             Carroll LWDA            176677                31122   
4           Frederick LWDA            302883               114511   
5              Howard LWDA            339183               189821   
6         Lower Shore LWDA            186331                70682   
7                 Maryland           6265347              3428701   
8          Montgomery LWDA           1074582               655253   
9     Prince George's LWDA            970374               867704   
10  Southern Maryland LWDA            387769               188516   
11        Susquehanna LWDA            373577               103341   
12        Upper Shore LWDA            179995                48279   
13   Western Maryland LWDA        

### Merging Wage Data with Demographics

In [29]:
# Merge on standard 'area_name'
df_mergedata = pd.merge(
    df_clean,
    df_lwda[
        ["area_name", "total_population", "minority_population", "minority_pct"]
    ],
    on="area_name",
    how="left",
)

# Export file
df_mergedata.to_csv("MD_Wage_Demographics_Merged.csv", index=False)

print(f"Merged dataset shape: {df_mergedata.shape}")
print(
    f"Unique LWDAs merged: {df_mergedata['area_name'].nunique()} / 14 (Statewide + 13 LWDAs)"
)
print("Missing values in merged dataset:\n", df_mergedata.isna().sum())

Merged dataset shape: (6864, 19)
Unique LWDAs merged: 14 / 14 (Statewide + 13 LWDAs)
Missing values in merged dataset:
 area_name                  0
occupation_code            0
occupation_name            0
occupation_group           0
employment                96
annual_mean_wage           0
annual_entry_wage          0
annual_experience_wage    30
annual_10th_percentile     0
annual_25th_percentile     0
annual_median_wage         0
annual_75th_percentile     0
annual_90th_percentile     0
is_total                   0
is_major_group             0
is_detailed                0
total_population           0
minority_population        0
minority_pct               0
dtype: int64


In [30]:
# Download clean dataset
# Comment out if already ran and downloaded

from google.colab import files
files.download("MD_Wage_Demographics_Merged.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>